# 06 Dashboard (Minimal MVP) — Interactive HTML Outputs

Components:
1) KPI cards (4)
2) risk_score distribution (boxplot, video-level)
3) bootstrap forest plot (delta + CI)
4) bootstrap table (sortable/filterable)
5) top risky videos table (sortable/filterable)
6) scatter: net_negativity vs engagement (size = n_threads_used)

Outputs:
- Data/data_06_dashboard_minimal_YYYYMMDD_HHMMSS/
  - 01_kpi_cards.html
  - 02_risk_score_boxplot.html
  - 03_bootstrap_forest.html
  - 04_bootstrap_table.html
  - 05_top_risky_videos_table.html
  - 06_scatter_netneg_vs_engagement.html
  - index.html  (links to all above)
  - dashboard_report.json


In [45]:
# Cell 1 — Imports
# ------------------------------------------

import os
import glob
from datetime import datetime

import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go


In [46]:
# Cell 2 — Locate 05 outputs (Data-only, latest folder)
# ------------------------------------------

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_ROOT = os.path.join(PROJECT_ROOT, "Data")

def pick_latest_dir(pattern: str) -> str:
    candidates = sorted(glob.glob(os.path.join(DATA_ROOT, pattern)), reverse=True)
    if not candidates:
        return ""
    return candidates[0]

LATEST_05_DIR = pick_latest_dir("data_05_scorecard_*")
if not LATEST_05_DIR:
    raise FileNotFoundError(f"❌ Cannot find data_05_scorecard_* under: {DATA_ROOT}. Please run Module 05 first.")

CELL_SUMMARY_PATH = os.path.join(LATEST_05_DIR, "scorecard_cell_summary.csv")
SCORECARD_VIDEO_PATH = os.path.join(LATEST_05_DIR, "scorecard_video.csv")
BOOTSTRAP_PATH = os.path.join(LATEST_05_DIR, "bootstrap_results.csv")
VIDEO_METRICS_PATH = os.path.join(LATEST_05_DIR, "video_metrics.csv")

for p in [CELL_SUMMARY_PATH, SCORECARD_VIDEO_PATH, BOOTSTRAP_PATH, VIDEO_METRICS_PATH]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"❌ Missing required file: {p}")

print("✅ Using 05 folder:", LATEST_05_DIR)


✅ Using 05 folder: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_05_scorecard_20260228_145858


In [47]:
# Cell 3 — Create 06 output folder (single folder per run)
# ------------------------------------------

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = os.path.join(DATA_ROOT, f"data_06_dashboard_html_{timestamp}")
os.makedirs(RUN_DIR, exist_ok=True)

def out(name: str) -> str:
    return os.path.join(RUN_DIR, name)

def save_plotly(fig, filename: str, title: str = None):
    if title:
        fig.update_layout(title=title)
    path = out(filename)
    fig.write_html(path, include_plotlyjs="cdn", full_html=True)
    print("✅ Saved:", path)

print("✅ 06 RUN_DIR:", RUN_DIR)


✅ 06 RUN_DIR: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957


In [48]:
# Cell 4 — Load tables
# ------------------------------------------

cell_summary = pd.read_csv(CELL_SUMMARY_PATH)
scorecard_video = pd.read_csv(SCORECARD_VIDEO_PATH)
bootstrap_df = pd.read_csv(BOOTSTRAP_PATH)
video_metrics = pd.read_csv(VIDEO_METRICS_PATH)

print("✅ loaded:")
print("  cell_summary", cell_summary.shape)
print("  scorecard_video", scorecard_video.shape)
print("  bootstrap_df", bootstrap_df.shape)
print("  video_metrics", video_metrics.shape)


✅ loaded:
  cell_summary (4, 14)
  scorecard_video (149, 18)
  bootstrap_df (12, 9)
  video_metrics (149, 14)


In [49]:
# Cell 5 — Save Plotly HTML helper
# ------------------------------------------
# Business-friendly names
METRIC_LABELS = {
    "negative_share": "Negative Share (Neg + Strong Neg)",
    "net_negativity": "Net Negative (Neg − Pos)",
    "engagement_index_mean": "Amplification Index (Replies + Likes, low-weighted)",
    "risk_score": "Risk Score (Composite)"
}

CELL_DISPLAY_ORDER = [
    "shein_labor",
    "non_shein_labor",
    "shein_environment",
    "non_shein_environment"
]

def minmax_01(s: pd.Series) -> pd.Series:
    s = s.astype(float)
    mn, mx = s.min(), s.max()
    if (mx - mn) < 1e-12:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - mn) / (mx - mn)

def format_delta_ci(delta, lo, hi, digits=3):
    return f"Δ={delta:.{digits}f}  CI[{lo:.{digits}f},{hi:.{digits}f}]"



In [50]:
# Cell 6 - (1) KPI cards（4个）
# -----------------------------------------------------------
# 提取 05 中你设定的权重，还原真实的成分比例
weights = {
    "negative_share": 0.45,
    "net_negativity": 0.35,
    "engagement_index_mean": 0.20
}

# 聚合每个阵营下三个驱动指标在 mm 化后的平均值
df_comp = scorecard_video.groupby("cell_id").agg(
    neg_mm_mean=("negative_share_mm", "mean"),
    netneg_mm_mean=("net_negativity_mm", "mean"),
    amp_mm_mean=("engagement_index_mean_mm", "mean"),
    n_videos=("video_id", "count"),
    mean_risk=("risk_score", "mean")
).reset_index()

# 计算真实加权贡献度（相加即等于 mean_risk）
df_comp["neg_contrib"] = df_comp["neg_mm_mean"] * weights["negative_share"]
df_comp["netneg_contrib"] = df_comp["netneg_mm_mean"] * weights["net_negativity"]
df_comp["amp_contrib"] = df_comp["amp_mm_mean"] * weights["engagement_index_mean"]

# 🌟 排序：风险高的排在上面 (后续靠 reversed 反转 Y 轴)
df_comp = df_comp.sort_values("mean_risk", ascending=False).reset_index(drop=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=df_comp["cell_id"], x=df_comp["neg_contrib"],
    name="Negative Share (45%)", orientation="h",
    hovertemplate="cell=%{y}<br>Contrib to Risk=%{x:.3f}<extra></extra>",
    marker_color="#EF553B"  # 橘红
))

fig.add_trace(go.Bar(
    y=df_comp["cell_id"], x=df_comp["netneg_contrib"],
    name="Net Negative (35%)", orientation="h",
    hovertemplate="cell=%{y}<br>Contrib to Risk=%{x:.3f}<extra></extra>",
    marker_color="#636EFA"  # 深紫蓝
))

fig.add_trace(go.Bar(
    y=df_comp["cell_id"], x=df_comp["amp_contrib"],
    name="Amplification (20%)", orientation="h",
    hovertemplate="cell=%{y}<br>Contrib to Risk=%{x:.3f}<extra></extra>",
    marker_color="#00CC96"  # 薄荷绿
))

# 标注总风险分
for i, r in df_comp.iterrows():
    fig.add_annotation(
        x=r["mean_risk"] + 0.02, y=r["cell_id"],
        text=f"Risk={r['mean_risk']:.3f} (n={int(r['n_videos'])})",
        showarrow=False, xanchor="left", font=dict(size=12, color="#333", family="Arial")
    )

fig.update_layout(
    barmode="stack",
    yaxis=dict(autorange="reversed"),  # 🌟 核心修复：保证高风险在最上方
    title="Risk Score Drivers (Actual Weighted Contribution)",
    xaxis_title="Contribution to Overall Risk Score (Sum = Risk)",
    yaxis_title="",
    legend_title="Risk components",
    height=520, margin=dict(l=40, r=120, t=70, b=30),
    plot_bgcolor="#f9f9f9", paper_bgcolor="#ffffff"
)

save_plotly(fig, "01_group_risk_drivers_stacked.html")

✅ Saved: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\01_group_risk_drivers_stacked.html


In [51]:
# Cell 7 - (2) risk_score 分布小提琴+箱线图
# ----------------------------------------------
BUSINESS_PALETTE = {
    'shein_labor': '#E63946', 'shein_environment': '#F4A261',
    'non_shein_labor': '#457B9D', 'non_shein_environment': '#2A9D8F'
}

fig = px.violin(
    scorecard_video,
    x="cell_id", y="risk_score", color="cell_id",
    box=True,      # 🌟 开启内部箱线图
    points=False,  # 🌟 隐藏零散数据点
    hover_data=["video_id"],
    color_discrete_map=BUSINESS_PALETTE,
    title="Risk Score Distribution (Violin + Inner Boxplot)"
)

# 添加均值标注
means = scorecard_video.groupby("cell_id")["risk_score"].mean().to_dict()
for cell, m in means.items():
    fig.add_annotation(
        x=cell, y=m, text=f"mean={m:.3f}",
        showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=1.5,
        yshift=10, font=dict(color="#333")
    )

fig.update_layout(
    xaxis_title="Group (cell_id)", yaxis_title="Risk Score (Composite, video-level)",
    legend_title="Group", height=560,
    margin=dict(l=40, r=40, t=70, b=40),
    plot_bgcolor="#f9f9f9", paper_bgcolor="#ffffff"
)

fig.update_traces(meanline_visible=True) # 小提琴内部显示均值线

save_plotly(fig, "02_risk_score_distribution_violin.html")

✅ Saved: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\02_risk_score_distribution_violin.html


In [52]:
# Cell 8 - Focused forest (only 2 contrasts) — business readable
# ---------------------------------------------------------------
primary = bootstrap_df[(bootstrap_df["groupA"] == "shein_labor") & (bootstrap_df["groupB"] == "non_shein_labor")].copy()
primary = primary[primary["metric"].isin(["negative_share", "net_negativity", "engagement_index_mean"])].copy()

primary["metric_label"] = primary["metric"].map(METRIC_LABELS)
primary["metric"] = pd.Categorical(primary["metric"], categories=["negative_share", "net_negativity", "engagement_index_mean"], ordered=True)
primary = primary.sort_values("metric")

fig = go.Figure()
x, lo, hi = primary["delta_hat"].astype(float).values, primary["ci_low"].astype(float).values, primary["ci_high"].astype(float).values
y = primary["metric_label"].tolist()

fig.add_trace(go.Bar(
    x=x, y=y, orientation="h",
    hovertext=[f"{r.metric_label}<br>{format_delta_ci(r.delta_hat, r.ci_low, r.ci_high)}<br>P(Δ>0)={r.prob_delta_gt0:.3f}" for r in primary.itertuples()],
    hoverinfo="text",
    marker_color="#E63946", opacity=0.85 # 加深颜色
))

fig.update_traces(error_x=dict(type="data", symmetric=False, array=(hi - x), arrayminus=(x - lo), color="#333", thickness=2))
fig.add_vline(x=0, line_width=2, line_dash="dash", line_color="gray")

# 🌟 核心防重叠设计：yshift 强行上推
for r in primary.itertuples():
    fig.add_annotation(
        x=float(r.delta_hat), y=r.metric_label,
        text=format_delta_ci(r.delta_hat, r.ci_low, r.ci_high, digits=3),
        showarrow=False, xanchor="center", yshift=22, # 上浮 22 像素
        font=dict(size=13, color="#111", family="Arial")
    )

fig.update_layout(
    title="How much worse is SHEIN-Labor vs peers? (shein_labor − non_shein_labor, 95% CI)",
    xaxis_title="Delta (higher = worse for risk)", yaxis_title="Business metric",
    height=460, margin=dict(l=40, r=40, t=70, b=30),
    plot_bgcolor="#f9f9f9", paper_bgcolor="#ffffff"
)

save_plotly(fig, "03_primary_contrast_delta_ci.html")

✅ Saved: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\03_primary_contrast_delta_ci.html


In [ ]:
# Cell 9 - (4) 核心四大对比阵营 (Times New Roman + 双向渐变色阶 + 单元格精细框线)
# -------------------------------------------------------------------
import plotly.graph_objects as go
import pandas as pd
import os

TARGET_CONTRASTS = [
    ("shein_labor", "non_shein_labor"),             # 1
    ("shein_labor", "shein_environment"),           # 2
    ("shein_environment", "non_shein_environment"), # 3
    ("non_shein_labor", "non_shein_environment")    # 4
]

core_metrics = ["negative_share", "net_negativity", "engagement_index_mean"]

# 🌟 1. 升级版：双向渐变色阶 (红绿双轨)
def get_delta_color(val, min_val, max_val):
    val = float(val)
    if val == 0: 
        return "#ffffff" # 0绝对留白
    elif val > 0:
        # 正数：红色渐变 (越接近最大值，红色越深)
        safe_max = max_val if max_val > 0 else 1
        alpha = min(0.8, max(0.1, (val / safe_max) * 0.8))
        return f"rgba(230, 57, 70, {alpha})"
    else:
        # 负数：绿色渐变 (越接近最小值/负得越多，绿色越深)
        safe_min = abs(min_val) if min_val < 0 else 1
        alpha = min(0.8, max(0.1, (abs(val) / safe_min) * 0.8))
        return f"rgba(42, 157, 143, {alpha})"

def get_ci_color(low, high):
    if float(low) <= 0 <= float(high): return "rgba(42, 157, 143, 0.2)" # 跨0标绿
    else: return "rgba(230, 57, 70, 0.2)" # 不跨0标红

def get_p_color(p):
    if float(p) > 0.5: return "rgba(230, 57, 70, 0.2)" # >50%标红
    else: return "rgba(42, 157, 143, 0.2)" # <=50%标绿

html_snippets = []
BORDER_COLOR = "#d1d5db" 

for i, (group_a, group_b) in enumerate(TARGET_CONTRASTS, start=1):
    df_sub = bootstrap_df[
        (bootstrap_df["metric"].isin(core_metrics)) & 
        (bootstrap_df["groupA"] == group_a) & 
        (bootstrap_df["groupB"] == group_b)
    ].copy()
    
    if df_sub.empty: continue

    df_sub["Delta_Str"] = df_sub["delta_hat"].apply(lambda x: f"{float(x):+.3f}")
    df_sub["CI_Str"] = df_sub.apply(lambda r: f"[{float(r.ci_low):.3f}, {float(r.ci_high):.3f}]", axis=1)
    df_sub["Prob_Str"] = df_sub["prob_delta_gt0"].apply(lambda x: f"{float(x):.1%}")
    df_sub["metric_cat"] = pd.Categorical(df_sub["metric"], categories=core_metrics, ordered=True)
    df_sub = df_sub.sort_values("metric_cat")
    
    # 🌟 2. 同时提取最大值和最小值，喂给双向变色函数
    max_delta = df_sub["delta_hat"].max()
    min_delta = df_sub["delta_hat"].min()
    
    col1_colors = ["#ffffff"] * len(df_sub)
    col2_colors = ["#ffffff"] * len(df_sub)
    col3_colors = ["#ffffff"] * len(df_sub)
    col4_colors = [get_delta_color(x, min_delta, max_delta) for x in df_sub["delta_hat"]]
    col5_colors = [get_ci_color(l, h) for l, h in zip(df_sub["ci_low"], df_sub["ci_high"])]
    col6_colors = [get_p_color(p) for p in df_sub["prob_delta_gt0"]]
    
    fill_colors = [col1_colors, col2_colors, col3_colors, col4_colors, col5_colors, col6_colors]
    
    fig = go.Figure(data=[go.Table(
        header=dict(
            values=["<b>metric</b>", "<b>groupA</b>", "<b>groupB</b>", "<b>Δ</b>", "<b>95% CI</b>", "<b>p(Δ>0)</b>"],
            fill_color="#2a3f5f", 
            align=["left", "left", "left", "center", "center", "center"],
            font=dict(color="white", size=14, family="Times New Roman"),
            height=35,
            line=dict(color=BORDER_COLOR, width=1) 
        ),
        cells=dict(
            values=[df_sub["metric"], df_sub["groupA"], df_sub["groupB"], df_sub["Delta_Str"], df_sub["CI_Str"], df_sub["Prob_Str"]],
            fill_color=fill_colors,
            align=["left", "left", "left", "center", "center", "center"],
            font=dict(color="#333", size=14, family="Times New Roman"),
            height=30,
            line=dict(color=BORDER_COLOR, width=1) 
        )
    )])
    
    fig.update_layout(
        title=dict(text=f"Contrast {i}: {group_a} vs {group_b}", font=dict(size=16, color="#2a3f5f", family="Times New Roman", weight="bold")),
        height=220, margin=dict(l=20, r=20, t=40, b=10)
    )
    
    include_js = 'cdn' if len(html_snippets) == 0 else False
    html_snippets.append(fig.to_html(full_html=False, include_plotlyjs=include_js))

final_html = f"""
<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Bootstrap Contrasts Matrix</title>
    <style>
        body {{ font-family: 'Times New Roman', Times, serif; background-color: #f0f2f6; padding: 30px; margin: 0; }}
        .dashboard-header {{ text-align: center; margin-bottom: 30px; color: #1f2937; }}
        .table-card {{ background-color: white; border-radius: 12px; padding: 15px; margin-bottom: 25px; box-shadow: 0 4px 6px rgba(0,0,0,0.05); border: 1px solid #e5e7eb; }}
    </style>
</head>
<body>
    <div class="dashboard-header">
        <h2>Data Science Report: Bootstrap Resampling Contrasts</h2>
        <p style="color: #6b7280;">
            <b>Delta:</b> <span style="color:#E63946;">Red gradient (A > B)</span> | <span style="color:#2A9D8F;">Green gradient (A < B)</span><br>
            <b>CI:</b> <span style="color:#E63946;">Red (Does not cross 0)</span> | <span style="color:#2A9D8F;">Green (Crosses 0)</span><br>
            <b>P-value:</b> <span style="color:#E63946;">Red (>50%)</span> | <span style="color:#2A9D8F;">Green (≤50%)</span>
        </p>
    </div>
    {''.join([f'<div class="table-card">{snippet}</div>' for snippet in html_snippets])}
</body>
</html>
"""

output_path = os.path.join(RUN_DIR, "04_bootstrap_tables_combined.html")
with open(output_path, "w", encoding="utf-8") as f:
    f.write(final_html)

print(f"✅ 带有双向发散色阶的终极表格生成完毕: {output_path}")

✅ 带有双向发散色阶的终极表格生成完毕: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\04_bootstrap_tables_combined.html


In [54]:
# Cell 10 - (5) 散点图：net_negativity vs engagement（size = n_threads_used）
# -------------------------------------------------------------------
df = video_metrics.copy()
x_med = float(df["net_negativity"].median())
y_med = float(df["engagement_index_mean"].median())

BUSINESS_PALETTE = {
    'shein_labor': '#E63946', 'shein_environment': '#F4A261',
    'non_shein_labor': '#457B9D', 'non_shein_environment': '#2A9D8F'
}

fig = px.scatter(
    df, x="net_negativity", y="engagement_index_mean",
    color="cell_id", size="n_threads_used",
    hover_data=["video_id", "negative_share", "pos_share", "replies_sum"] if "pos_share" in df.columns else ["video_id", "n_threads_used"],
    color_discrete_map=BUSINESS_PALETTE,
    opacity=0.75, # 透明气泡防重叠
    title="Which videos fall into the high-risk viral zone? (Net Negative × Amplification)"
)

# 🌟 气泡描边白框提升质感
fig.update_traces(marker=dict(line=dict(width=1, color="White")))

# 十字中轴
fig.add_vline(x=x_med, line_width=2, line_dash="dash", line_color="#888")
fig.add_hline(y=y_med, line_width=2, line_dash="dash", line_color="#888")

# 指引文字
fig.add_annotation(x=x_med, y=df["engagement_index_mean"].max(), text="More net negative →", showarrow=False, yshift=12, font=dict(color="#666"))
fig.add_annotation(x=df["net_negativity"].min(), y=y_med, text="↑ More amplification", showarrow=False, xshift=12, textangle=-90, font=dict(color="#666"))

x_min, x_max = float(df["net_negativity"].min()), float(df["net_negativity"].max())
y_min, y_max = float(df["engagement_index_mean"].min()), float(df["engagement_index_mean"].max())

# 🌟 带有半透明磨砂背景板的象限解释
fig.add_annotation(
    x=(x_med + x_max)/2, y=(y_med + y_max)/2,
    text="<b>High-risk viral zone</b><br><i>(high net neg + high amp)</i>",
    showarrow=False, font=dict(size=13, color="#E63946"),
    bgcolor="rgba(255,255,255,0.75)", bordercolor="#E63946", borderwidth=1, borderpad=4
)
fig.add_annotation(
    x=(x_min + x_med)/2, y=(y_med + y_max)/2,
    text="<b>Heated but balanced</b><br><i>(low net neg + high amp)</i>",
    showarrow=False, font=dict(size=12, color="#457B9D"),
    bgcolor="rgba(255,255,255,0.75)", bordercolor="#457B9D", borderwidth=1, borderpad=4
)
fig.add_annotation(
    x=(x_med + x_max)/2, y=(y_min + y_med)/2,
    text="<b>Negative but contained</b><br><i>(high net neg + low amp)</i>",
    showarrow=False, font=dict(size=12, color="#F4A261"),
    bgcolor="rgba(255,255,255,0.75)", bordercolor="#F4A261", borderwidth=1, borderpad=4
)
fig.add_annotation(
    x=(x_min + x_med)/2, y=(y_min + y_med)/2,
    text="<b>Low concern zone</b>",
    showarrow=False, font=dict(size=12, color="#2A9D8F"),
    bgcolor="rgba(255,255,255,0.75)", bordercolor="#2A9D8F", borderwidth=1, borderpad=4
)

fig.update_layout(
    xaxis_title="Net Negative (Neg − Pos): lack of defense / consensus risk",
    yaxis_title="Amplification Index: replies + likes (low-weighted)",
    height=650, margin=dict(l=40, r=40, t=70, b=40),
    plot_bgcolor="#f9f9f9", paper_bgcolor="#ffffff",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor="rgba(255,255,255,0.85)", bordercolor="#ccc", borderwidth=1)
)

save_plotly(fig, "05_quadrant_netneg_vs_amplification.html")

✅ Saved: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\05_quadrant_netneg_vs_amplification.html


In [55]:
# Cell 12 - Print outputs
# -------------------------------------------------------------------

print("\n" + "="*60)
print("✅ 06 HTML-only outputs generated")
print("="*60)
print("Output folder:", RUN_DIR)
print("Files:")
for fn in [
    "01_kpi_cards.html",
    "02_risk_score_boxplot.html",
    "03_bootstrap_forest.html",
    "04_bootstrap_table.html",
    "05_top_risky_videos_table.html",
    "06_scatter_netneg_vs_engagement.html",
]:
    print(" -", os.path.join(RUN_DIR, fn))



✅ 06 HTML-only outputs generated
Output folder: d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957
Files:
 - d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\01_kpi_cards.html
 - d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\02_risk_score_boxplot.html
 - d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\03_bootstrap_forest.html
 - d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\04_bootstrap_table.html
 - d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\05_top_risky_videos_table.html
 - d:\coding\projects\Python\Youtube Comment Sentiment Analysis\Data\data_06_dashboard_html_20260228_145957\06_scatter_netneg_vs_engagement.html
